In [22]:
import json
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import operator
from statistics import mean
from collections import Counter

In [23]:
with open('data/train-claims.json', 'r') as input_file:
    train_claim_data = json.load(input_file)


with open('data/dev-claims.json', 'r') as input_file:
    dev_claim_data = json.load(input_file)


with open('data/test-claims-unlabelled.json', 'r') as input_file:
    test_claim_data = json.load(input_file)


with open('data/evidence.json', 'r') as input_file:
    evi_data = json.load(input_file)

In [24]:

full_evidence_id = list(evi_data.keys())
full_evidence_text  = list(evi_data.values())
train_claim_id = list(train_claim_data.keys())
train_claim_text  = [ v["claim_text"] for v in train_claim_data.values()]

In [25]:

# dev remove stop word会好一点， test不remove反而好
evidence_tfidf_vectorizer = TfidfVectorizer(stop_words="english", use_idf=True)

# claim_tfidf_vectorizer = TfidfVectorizer(stop_words="english", use_idf=True)
claim_tfidf_vectorizer = TfidfVectorizer(use_idf=True)


evidence_tfidf_vectorizer.fit(train_claim_text+full_evidence_text)
train_claim_emb_list = claim_tfidf_vectorizer.fit_transform(train_claim_text)


full_evi_emb_list = evidence_tfidf_vectorizer.transform(full_evidence_text)


In [26]:
# with open('data/test-claims-unlabelled.json', 'r') as input_file:
#     test_out_temp = json.load(input_file)

with open('data/dev-claims.json', 'r') as input_file:
    test_out_temp2 = json.load(input_file)

In [27]:

evi_k=2 # 最相似的evidence数量
claim_k=1 # 最相似的claim数量

for claim_id,claim_value in test_out_temp2.items():

    # Retrival evidence
    # test claim convert to vector
    
    test_claim_emb = evidence_tfidf_vectorizer.transform([claim_value['claim_text']])
    evi_sim_dict = {}

    # Calculate similarity between test claim and evidence
    
    sim = cosine_similarity(test_claim_emb, full_evi_emb_list)[0]
    
    for i in range(len(sim)):
        evi_sim_dict[full_evidence_id[i]] = sim[i]
    
    # 找出最相似的k个
    s_sim = [(k, v) for k, v in sorted(evi_sim_dict.items(), key=lambda item: item[1],reverse=True)][:evi_k]
    sel_sim = [k for k,v in s_sim]

    # 把最相似的前k个evidence的id写入到test claim的evidence list
    test_out_temp2[claim_id]["evidences"] = sel_sim
   
    # Classification 
    # 通过与当前test claim 最相似的train claims, 然后再把对应得label拿出来

    # test claim convert to vector with stopword
    test_claim_emb = claim_tfidf_vectorizer.transform([claim_value['claim_text']])

    # 计算出test claim和所有train claim的相似度
    
    claim_sim_dict = {}
    claim_sim = cosine_similarity(test_claim_emb, train_claim_emb_list)[0]
    for i in range(len(claim_sim)):
        claim_sim_dict[train_claim_id[i]] = claim_sim[i]
    
    # 取最相似的k个train claim
    most_sim_claims = [(k, v) for k, v in sorted(claim_sim_dict.items(), key=lambda item: item[1],reverse=True)]
    # 取最相似的那一个
    most_sim_claim = max(most_sim_claims, key=operator.itemgetter(1))[0]
    
    test_out_temp2[claim_id]["claim_label"] = train_claim_data[most_sim_claim]["claim_label"]

# Writing to sample.json
# with open("data/test_predict_knn.json", "w") as outfile:
#     json.dump(test_out_temp, outfile)


with open("data/dev_predict_knn3.json", "w") as outfile:
    json.dump(test_out_temp2, outfile)

KeyboardInterrupt: 

In [ ]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import KeyedVectors
from nltk.corpus import stopwords
from gensim.similarities import WmdSimilarity

# Load pre-trained Word2Vec model
# model = KeyedVectors.load_word2vec_format('word2vec.model', binary=True)
model = KeyedVectors.load('word2vec.wordvectors', mmap='r')

# Stop words list
stop_words = set(stopwords.words('english'))

# Load your data structures: full_evidence_id, full_evi_emb_list, train_claim_id, train_claim_emb_list, etc.
# Assuming these are loaded/prepared elsewhere in your code

# Number of top similar items to retrieve
evi_k = 2
claim_k = 1

# Use WMD to find similarities and retrieve top k items
for claim_id, claim_value in test_out_temp2.items():
    # Convert test claim to lowered words, ignoring stopwords
    test_claim_words = [word for word in claim_value['claim_text'].lower().split() if word not in stop_words]

    # Setup WMD similarity
    instance = WmdSimilarity(full_evi_emb_list, model, num_best=evi_k)

    # Find the most similar evidences using WMD
    sims = instance[test_claim_words]
    selected_evidence_ids = [full_evidence_id[i] for i, score in sims]

    # Store the most similar evidences in the output
    test_out_temp2[claim_id]["evidences"] = selected_evidence_ids

    # Repeat process for claim similarity
    claim_instance = WmdSimilarity(train_claim_emb_list, model, num_best=claim_k)
    claim_sims = claim_instance[test_claim_words]
    most_sim_claim_id = train_claim_id[claim_sims[0][0]]

    # Store the label of the most similar claim
    test_out_temp2[claim_id]["claim_label"] = train_claim_data[most_sim_claim_id]["claim_label"]

# Write output to file
with open("data/dev_predict_knn3.json", "w") as outfile:
    json.dump(test_out_temp2, outfile)


TypeError: sparse matrix length is ambiguous; use getnnz() or shape[0]

In [ ]:

import subprocess

output = subprocess.check_output("python eval.py --predictions data/dev_predict_knn3.json --groundtruth data/dev-claims.json", shell=True)

print(output)

b'Evidence Retrieval F-score (F)    = 0.09755720470006184\r\nClaim Classification Accuracy (A) = 0.474025974025974\r\nHarmonic Mean of F and A          = 0.16181249099831735\r\n'
